# Phase 4: zero-shot Qwen3-ASR-0.6B and Reson8 Rumble-2-Flash on dev

Two more zero-shot baselines, through the same `cyasr.run_baseline` script and scoring as the Whisper runs:

- `Qwen/Qwen3-ASR-0.6B-hf` (Apache-2.0, Greek is one of its 30 languages).
- `Reson8/Rumble-2-Flash`, Reson8's fine-tune of Qwen3-ASR-0.6B (Apache-2.0, 45 languages incl. Greek), run twice:
  with its product defaults (numbers as digits, fillers dropped) and with options matched to our references
  (`digits: false, filler: true`). The difference shows how much is transcription convention, not recognition.

Both use the native transformers implementation (no `qwen-asr` package, which would downgrade transformers to
4.57). Whole clips go through in one pass (limit 20 min, our longest is 271 s): no 30 s windows, so no
silent-tail or window-boundary effects, unlike Whisper. Greedy decoding, Greek forced, pinned model revisions.

Use an **L4** runtime (bfloat16). On a T4 the backend falls back to float32 (slower, but it fits).

Before starting, run `bash scripts/make_colab_bundle.sh` locally and upload the new `cyasr_bundle.zip` to
`MyDrive/cyasr/`, replacing the old one (it must contain `cyasr/transcribe_qwen.py`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
BUNDLE = '/content/drive/MyDrive/cyasr/cyasr_bundle.zip'
DRIVE_RUNS = '/content/drive/MyDrive/cyasr/runs'
CONFIGS = ['configs/baseline_qwen3-asr-0.6b.yaml',
           'configs/baseline_rumble-2-flash.yaml',
           'configs/baseline_rumble-2-flash_matched.yaml']

!rm -rf /content/cyasr && mkdir -p /content/cyasr
!unzip -q {BUNDLE} -d /content/cyasr
%cd /content/cyasr
!ls cyasr/transcribe_qwen.py {' '.join(CONFIGS)}  # fails if the bundle is the old one
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q -r requirements.txt
# Colab ships an old torchao that peft rejects ("only versions above 0.16.0"); we don't use it
!pip uninstall -y -q torchao
import transformers; print(transformers.__version__)  # Qwen3-ASR needs a recent 5.x (requirements pin 5.22.0)

In [ ]:
# Smoke test: 4 dev clips per model, saved locally. Checks that each model loads and writes Greek text.
import json, yaml
for cfg_path in CONFIGS:
    !python -m cyasr.run_baseline --config {cfg_path} --max-samples 4
    name = yaml.safe_load(open(cfg_path))['run_name']
    for line in open(f'runs/{name}_greedy_dev_n4/predictions.jsonl', encoding='utf-8'):
        r = json.loads(line)
        print(f"--- {r['audio_id']} ({r['duration_s']:.0f} s, {r['n_tokens']} tokens)\nREF: {r['ref'][:200]}\nHYP: {r['hyp'][:200]}")

In [ ]:
# Does batch size change the output? Whisper's did (padding), which we fixed by batching short and long
# clips apart. Here clips are sorted by length and padding is masked, but check: 8 clips, batch 1 vs 8.
# Result on 2026-11-20 (L4, bf16): 0/8 identical; batch 1 run twice: 8/8 identical. So the configs use batch 1.
import subprocess
def run_with(batch_size, out):
    cfg = yaml.safe_load(open(CONFIGS[0]))
    cfg.update(batch_size=batch_size, max_samples=8, output_dir=out)
    yaml.safe_dump(cfg, open('/content/bs.yaml', 'w'), allow_unicode=True)
    subprocess.run(['python', '-m', 'cyasr.run_baseline', '--config', '/content/bs.yaml'], check=True)
    name = f"{cfg['run_name']}_greedy_dev_n8"
    return [json.loads(l)['hyp'] for l in open(f'{out}/{name}/predictions.jsonl', encoding='utf-8')]
a, b = run_with(1, 'runs_bs1'), run_with(8, 'runs_bs8')
same = sum(x == y for x, y in zip(a, b))
print(f'identical: {same}/{len(a)}')
for x, y in zip(a, b):
    if x != y:
        print('bs1:', x[:150]); print('bs8:', y[:150])

In [ ]:
# The real runs: all 280 dev clips, written straight to Drive. batch_size is 1 in the configs: in bf16 the
# output depends on the batch size (rounding, not padding: in fp32 batch 1 and 8 agree on 7 of 8 clips),
# while batch 1 is deterministic.
import time
for cfg_path in CONFIGS:
    cfg = yaml.safe_load(open(cfg_path))
    cfg.update(output_dir=DRIVE_RUNS)
    yaml.safe_dump(cfg, open('/content/eval.yaml', 'w'), allow_unicode=True)
    t0 = time.time()
    !python -m cyasr.run_baseline --config /content/eval.yaml
    print(f'>>> {cfg_path}: {(time.time() - t0) / 60:.1f} min')

In [ ]:
# All zero-shot models on dev, same clips and scoring. Whisper: fallback decoding (its best), rule off.
import os
def load(name):  # None if the run has not finished (no metrics.json)
    path = f'{DRIVE_RUNS}/{name}/metrics.json'
    return json.load(open(path)) if os.path.exists(path) else None

runs = [('whisper-small', 'baseline_whisper-small_fallback_dev'),
        ('whisper-medium', 'baseline_whisper-medium_fallback_dev'),
        ('whisper-large-v3', 'baseline_whisper-large-v3_fallback_dev'),
        ('qwen3-asr-0.6b', 'baseline_qwen3-asr-0.6b_greedy_dev'),
        ('rumble-2-flash', 'baseline_rumble-2-flash_greedy_dev'),
        ('rumble matched', 'baseline_rumble-2-flash_matched_greedy_dev')]
print(f"{'model':17} {'WER basic':>9} {'nofill':>7} {'nodiac':>7} {'phonetic':>8} {'CER nodiac':>10} {'halluc.':>7} {'empty':>5} {'cut off':>7}")
for label, name in runs:
    m = load(name)
    if m is None:
        print(f"{label:17} {'missing':>9}"); continue
    lv = m['levels']
    print(f"{label:17} {lv['basic']['wer']:9.1%} {lv['nofill']['wer']:7.1%} {lv['nodiac']['wer']:7.1%} "
          f"{lv['phonetic']['wer']:8.1%} {lv['nodiac']['cer']:10.1%} {m['hallucinations']:7} "
          f"{m['empty_hypotheses']:5} {m.get('hit_max_tokens', '-'):>7}")

In [ ]:
# WER (nodiac) by clip length. Qwen-family models see the whole clip at once; Whisper slides 30 s windows,
# and its 30-33 s clips suffer from the silent last window.
import jiwer
from cyasr.normalize import normalize

def bucket(d):
    return '<=30s' if d <= 30 else ('30-33s' if d <= 33 else '>33s')

print(f"{'model':17} {'all':>6} {'<=30s':>6} {'30-33s':>7} {'>33s':>6} {'insert.':>7} {'delet.':>7}")
for label, name in runs:
    if load(name) is None:
        continue
    recs = [json.loads(l) for l in open(f'{DRIVE_RUNS}/{name}/predictions.jsonl', encoding='utf-8')]
    errs, words, ins, dels = {}, {}, 0, 0
    for r in recs:
        ref, hyp = normalize(r['ref'], 'nodiac'), normalize(r['hyp'], 'nodiac')
        o = jiwer.process_words(ref, hyp)
        for b in (bucket(r['duration_s']), 'all'):
            errs[b] = errs.get(b, 0) + o.substitutions + o.deletions + o.insertions
            words[b] = words.get(b, 0) + len(ref.split())
        ins += o.insertions; dels += o.deletions
    w = {b: errs[b] / words[b] for b in errs}
    print(f"{label:17} {w['all']:6.1%} {w['<=30s']:6.1%} {w['30-33s']:7.1%} {w['>33s']:6.1%} {ins:7} {dels:7}")

In [ ]:
# Did the processor's repetition clean-up change anything? (hyp = official output, hyp_raw = before clean-up)
for label, name in runs[3:]:
    if load(name) is None:
        continue
    recs = [json.loads(l) for l in open(f'{DRIVE_RUNS}/{name}/predictions.jsonl', encoding='utf-8')]
    changed = [r for r in recs if r['hyp'] != r['hyp_raw']]
    print(f'{label}: repetition clean-up changed {len(changed)} of {len(recs)} clips; '
          f"max tokens {max(r['n_tokens'] for r in recs)}")

In [ ]:
# Five short clips side by side: reference, zero-shot whisper-large-v3, Qwen, Rumble (matched)
import random
def preds(name):
    return {r['audio_id']: r for r in map(json.loads, open(f'{DRIVE_RUNS}/{name}/predictions.jsonl', encoding='utf-8'))}
lv3, qw, rb = (preds(n) for n in ['baseline_whisper-large-v3_fallback_dev', 'baseline_qwen3-asr-0.6b_greedy_dev',
                                    'baseline_rumble-2-flash_matched_greedy_dev'])
short = sorted(a for a in qw if qw[a]['duration_s'] <= 20)
for a in random.Random(0).sample(short, 5):
    print(f"--- {a} ({qw[a]['duration_s']:.0f} s)")
    print('REF     :', qw[a]['ref'])
    print('LARGE-V3:', lv3[a]['hyp'])
    print('QWEN    :', qw[a]['hyp'])
    print('RUMBLE  :', rb[a]['hyp'])